## Imports


In [ ]:
import polars as pl
from pathlib import Path

# Data Path


In [ ]:
data_folder = Path("../data")

# Load Dataset


In [ ]:
sales_df = pl.read_parquet(
    data_folder / "sales_cleaned.parquet"
)

# Preview Dataset
sales_df.head()

## Create Year-Month Column


In [ ]:
# Extract year-month for aggregation
sales_df = sales_df.with_columns(
    pl.col("purchase_date").dt.strftime("%Y-%m").alias("year_month")
)

# Monthly Sales Trends


In [ ]:
monthly_sales = sales_df.group_by("year_month").agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue"),
    pl.col("price").mean().round().alias("average_order_value")
).sort("year_month")

monthly_sales

# Revenue Share by Month


In [ ]:
monthly_sales = monthly_sales.with_columns(
    (
        pl.col("total_revenue") / pl.col("total_revenue").sum() * 100
    ).round().alias("revenue_share_pct")
)

monthly_sales.sort("revenue_share_pct", descending=True)

# Festival Monthly Trends


In [ ]:
monthly_festival_sales = sales_df.group_by(["year_month", "festival"]
                                           ).agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["year_month", "total_revenue"], descending=[False, True])

monthly_festival_sales.head(10)

# Adding month column to sales_df


In [ ]:
sales_df = sales_df.with_columns(
    pl.col("purchase_date").dt.strftime("%B").alias("month_name")
)

# Save Outputs


In [ ]:
monthly_festival_sales.write_parquet(
    data_folder / "monthly_festival_sales.parquet")
monthly_sales.write_parquet(data_folder / "monthly_sales.parquet")

In [ ]:
monthly_sales

In [ ]:
monthly_festival_sales

In [ ]:
sales_df